<a href="https://colab.research.google.com/github/n5n0/eskwelabs-portfolio/blob/main/py7_tables_practice_bank_Crinan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# py7 · Tables: Python Practice Bank (Day 7)

**DABC Sprint 2 · AML case · the same five tables you queried in SQL on Days 1–4**

### How to use this bank

- Questions are tiered **EASY · AVERAGE · DIFFICULT**. Work in order.
- **In class:** Practice block 1 (18:50) → **Q1–10**, after the account behaviour table. Practice block 2 (19:50) → **Q11–25**. Your mentor will tell you the floor (what everyone finishes) for your room.
- **SQL twin** names the Day 1–4 question that asks the same thing. When both use the same table, your Python answer should give the same number as your SQL answer. If it doesn't, find out why: that's the lesson.
- Try each question yourself first. Then use AI to explain or check, following the sprint's AI protocol: paste the column names, never rows of data, and note it in your AI use log.
- ✍️ cells ask for a sentence, not code. Write it for the Head of AML.
- Your mentor has the answer key. Remaining questions are homework.

**Data you need** (Sprint 2 student folder, the same tables you queried in BigQuery on Days 1–4): `customers.csv`, `accounts.csv`, `transactions.csv`, `alerts.csv`, `investigations.csv` and `aml_analytical_extract.csv` (the Day 4 extract). Upload them to this Colab session, or put them in a Drive folder and add that folder to `DATA_DIRS` below.

## Step 0 · Setup

Run these four cells in order with **Shift + Enter**. After a runtime restart, run them again.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("pandas", pd.__version__, "· ready")

pandas 2.2.3 · ready


In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION — the only cell you should need to edit
# ---------------------------------------------------------------------------
USE_BIGQUERY = False                  # True = read the tables straight from BigQuery
PROJECT_ID   = "your-project-id"      # ← paste the project ID from your instructor
DATASET      = "aml_sprint2"

# Folders searched, in order, for the CSV files. Add your own Drive folder if needed.
DATA_DIRS = [
    ".",                                          # same folder as this notebook
    "data",
    "/content",                                   # Colab: files uploaded to this session
    "/content/drive/MyDrive/DABC24/aml_sprint2",  # Colab + Drive (mount Drive first)
]
print("USE_BIGQUERY =", USE_BIGQUERY)

USE_BIGQUERY = False


In [ ]:
def load(name):
    """Return one table as a DataFrame.

    name is one of: customers, accounts, transactions, alerts, investigations,
    aml_analytical_extract. Prints where the data came from, so you always know.
    """
    if USE_BIGQUERY:
        from google.colab import auth
        from google.cloud import bigquery
        auth.authenticate_user()
        client = bigquery.Client(project=PROJECT_ID)
        frame = client.query(
            f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{name}`"
        ).to_dataframe()
        print(f"{name:<24} {len(frame):>7,} rows  (BigQuery)")
        return frame

    for folder in DATA_DIRS:
        path = os.path.join(folder, f"{name}.csv")
        if os.path.exists(path):
            frame = pd.read_csv(path)
            print(f"{name:<24} {len(frame):>7,} rows  ({path})")
            return frame

    raise FileNotFoundError(
        f"{name}.csv not found in {DATA_DIRS}. Upload it to this session "
        "or add its folder to DATA_DIRS in the configuration cell."
    )

In [ ]:
customers      = load("customers")
accounts       = load("accounts")
transactions   = load("transactions")
alerts         = load("alerts")
investigations = load("investigations")
df             = load("aml_analytical_extract")

customers                 10,000 rows  (./customers.csv)
accounts                  12,028 rows  (./accounts.csv)
transactions              46,838 rows  (./transactions.csv)
alerts                     2,618 rows  (./alerts.csv)
investigations             1,623 rows  (./investigations.csv)
aml_analytical_extract    46,793 rows  (./aml_analytical_extract.csv)


## Setup · standard cleaning

Run this once. It gives you `df` with `channel_clean` and `occupation_clean`, the de-duplicated `accounts_d` and `transactions_d`, and `alerts["alert_score_num"]`.

In [ ]:
# Standard Day 6 cleaning, provided so this bank runs on its own.
# It uses the model decisions from the Day 6 cleaning log.
CHANNEL_MAP = {"atm": "ATM", "branch": "Branch", "instapay": "InstaPay",
               "mobile": "Mobile", "online": "Online", "pesonet": "PESONet"}
OCCUPATION_MAP = {"professional": "Professional", "business owner": "Business Owner",
                  "self-employed": "Business Owner", "student": "Student",
                  "unemployed": "Unemployed", "not employed": "Unemployed"}

df = df.copy()
df["channel_clean"] = df["channel"].str.strip().str.lower().map(CHANNEL_MAP)
df["occupation_clean"] = df["occupation"].str.strip().str.lower().map(OCCUPATION_MAP)

# De-duplicated source tables, same rule as 04_final_analytical_extract.sql
accounts_d = accounts.drop_duplicates(subset="acct_id", keep="first")
transactions_d = (transactions
                  .sort_values("tx_datetime", kind="stable")
                  .drop_duplicates(subset="tx_id", keep="first"))

# Numeric alert score: 'NOT_SCORED' becomes NaN
alerts["alert_score_num"] = pd.to_numeric(alerts["alert_score"], errors="coerce")

print(f"extract: {len(df):,} rows · accounts_d: {len(accounts_d):,} · "
      f"transactions_d: {len(transactions_d):,} · alerts: {len(alerts):,}")

extract: 46,793 rows · accounts_d: 12,000 · transactions_d: 46,793 · alerts: 2,618


## Block 1 · `GROUP BY` → `groupby().agg()` (Q1–10)

Named aggregation: `.agg(new_name=("column", "function"))`. Before each table, say what one row of the result is.

### Q1 · EASY · SQL twin: d3a Q6

Number of transactions per `channel_clean`, most first, including missing channels.

💡 *Hint:* `df.groupby("channel_clean", dropna=False).size()`.

In [ ]:
# Your code here
df.groupby("channel_clean", dropna=False).size().sort_values(ascending=False)

,0
channel_clean,
Online,8502
Branch,8477
PESONet,8014
Mobile,7233
InstaPay,7228
ATM,7199
NaN,140


### Q2 · EASY · SQL twin: d3a Q7–Q8

In one table: number of transactions, total amount and average amount per `channel_clean`, sorted by total amount.

💡 *Hint:* `.agg(transactions=("tx_id", "count"), total_amount=("amount", "sum"), avg_amount=("amount", "mean"))`.

In [ ]:
df.groupby("channel_clean", dropna=False).agg(
    transactions=("tx_id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean")
).sort_values("total_amount", ascending=False)

,transactions,total_amount,avg_amount
channel_clean,,,
Online,8502,"995,848,023.41","117,131.03"
Branch,8477,"920,921,013.62","108,637.61"
PESONet,8014,"799,428,999.89","99,754.06"
ATM,7199,"199,254,148.99","27,678.03"
InstaPay,7228,"176,497,642.19","24,418.60"
Mobile,7233,"150,266,284.59","20,775.10"
NaN,140,"8,326,595.63","59,475.68"


### Q3 · EASY · SQL twin: d3a Q10

Number of accounts per `acct_type`, using the de-duplicated `accounts_d`.

In [ ]:
# Your code here
accounts_d.groupby("acct_type").size()

,0
acct_type,
Business,1996
Checking,3792
Payroll,2696
Savings,3516


### Q4 · AVERAGE · SQL twin: d3a Q11

Earliest and latest `open_date` per account type. Convert `open_date` to datetimes first, without changing `accounts_d` itself.

💡 *Hint:* `accounts_d.assign(open_date=pd.to_datetime(accounts_d["open_date"]))` gives a converted copy.

In [ ]:
# Your code here
accounts_d.assign(open_date=pd.to_datetime(accounts_d["open_date"], format="mixed")).groupby("acct_type").agg(earliest=("open_date", "min"), latest=("open_date", "max"))

,earliest,latest
acct_type,,
Business,2015-03-05,2025-11-30
Checking,2015-02-28,2025-11-30
Payroll,2015-04-22,2025-11-30
Savings,2015-03-29,2025-11-30


### Q5 · AVERAGE · SQL twin: d3a Q13

For each alert rule: number of alerts, number of **scored** alerts, and the average, minimum and maximum `alert_score_num`.

In [ ]:
# Your code here
alerts.groupby("rule_name").agg(alert_count=("tx_id", "count"), scored_alert_count=("alert_score_num", "count"), avg_score=("alert_score_num", "mean"), min_score=("alert_score_num", "min"), max_score=("alert_score_num", "max"))

,alert_count,scored_alert_count,avg_score,min_score,max_score
rule_name,,,,,
Dormant_Reactivation,126,116,62.76,34.80,95.60
High_Velocity_Count,462,446,60.58,16.60,92.90
Large_Amount_CTR,1540,1441,54.94,16.70,95.40
Near_Threshold_Structuring,366,341,72.04,39.40,100.00
Rapid_Movement,124,117,76.28,42.30,100.00


✍️ **Why is "number of alerts" different from "number of scored alerts"?**

*Your answer:* There are alerts that are labeled "NOT SCORED" under the alert_score_num column

### Q6 · AVERAGE · SQL twin: d3a Q25, Q27

For each channel: total transactions, transactions of PHP 500,000 or more, and that share as a percentage, highest share first.

💡 *Hint:* The mean of a 0/1 flag is a rate: `large_rate=("large_amount_flag", "mean")`.

In [ ]:
# Your code here
df.groupby("channel_clean", dropna=False).agg(
    total_transactions=("tx_id", "count"),
    large_transactions=("amount", lambda s: (s >= 500000).sum()),
    large_rate=("amount", lambda s: (s >= 500000).mean() * 100)
).sort_values("large_rate", ascending=False)

,total_transactions,large_transactions,large_rate
channel_clean,,,
PESONet,8014,502,6.26
Online,8502,484,5.69
Branch,8477,472,5.57
NaN,140,3,2.14
InstaPay,7228,41,0.57
Mobile,7233,25,0.35
ATM,7199,13,0.18


### Q7 · AVERAGE · SQL twin: d3a Q17

The five channels with the highest total amount.

In [ ]:
# Calculate total transaction amount per channel and display the top 5
top_five_channels = df.groupby("channel_clean", dropna=False).agg(
    total_amount=("amount", "sum")
).sort_values("total_amount", ascending=False).head(5)

display(top_five_channels)

,total_amount
channel_clean,
Online,"995,848,023.41"
Branch,"920,921,013.62"
PESONet,"799,428,999.89"
ATM,"199,254,148.99"
InstaPay,"176,497,642.19"


### Q8 · AVERAGE · SQL twin: d3a Q32

For each customer in `accounts_d`: number of accounts, number of distinct account types, earliest and latest opening date. What is the most accounts any one customer holds, and how many customers hold that many?

In [ ]:
# Pre-convert open_date to datetime to allow min/max operations
accounts_temp = accounts_d.assign(open_date=pd.to_datetime(accounts_d["open_date"], format="mixed"))

# Group by customer and calculate aggregations
cust_accounts = accounts_temp.groupby("cust_id").agg(
    num_accounts=("acct_id", "count"),
    unique_acct_types=("acct_type", "nunique"),
    earliest_open=("open_date", "min"),
    latest_open=("open_date", "max")
)

# Find the maximum accounts and how many customers hold that many
max_accts = cust_accounts["num_accounts"].max()
num_cust_with_max = (cust_accounts["num_accounts"] == max_accts).sum()

print(f"Maximum accounts held by any one customer: {max_accts}")
print(f"Number of customers holding {max_accts} accounts: {num_cust_with_max}\n")
display(cust_accounts.head())

Maximum accounts held by any one customer: 6
Number of customers holding 6 accounts: 13



,num_accounts,unique_acct_types,earliest_open,latest_open
cust_id,,,,
CUST-0006932d,1,1,2022-08-18,2022-08-18
CUST-00085b82,1,1,2024-09-17,2024-09-17
CUST-000a2af6,1,1,2020-01-17,2020-01-17
CUST-000cf71a,1,1,2020-09-08,2020-09-08
CUST-000fed8e,1,1,2017-02-11,2017-02-11


### Q9 · AVERAGE

Alert rate by originating account status (`orig_acct_status`): transactions, alerted transactions, and alert rate as a percentage. Which status stands out?

In [ ]:
# Calculate transaction metrics grouped by originating account status
alert_rate_by_status = df.groupby("orig_acct_status", dropna=False).agg(
    transactions=("tx_id", "count"),
    alerted_transactions=("was_alerted", "sum"),
    alert_rate=("was_alerted", lambda s: s.mean() * 100)
).sort_values("alert_rate", ascending=False)

display(alert_rate_by_status)

,transactions,alerted_transactions,alert_rate
orig_acct_status,,,
Dormant,743,122,16.42
Closed,148,11,7.43
Active,45902,2248,4.90


### Q10 · DIFFICULT

Build the **account behaviour table**: one row per `orig_acct` with `tx_count`, `total_amount`, `max_amount`, `near_threshold_tx`, `active_days` (distinct `tx_date`), `first_tx`, `last_tx` and `external_share` (share of transactions to `EXT-` accounts).

Prove that `orig_acct` is unique in your table and that `total_amount` adds back to the extract total. How many accounts made **3 or more** near-threshold transactions?

💡 *Hint:* `external_share`: make a 0/1 column first (`1 - dest_is_internal`) and take its mean. For float totals, check `abs(a - b) < 0.01` rather than `==`.

In [ ]:
# Your code here


## Block 2 · `JOIN` → `merge()` (Q11–25)

Every merge gets three habits: rows before and after, `validate=` with the expected cardinality, and `indicator=True` when you need to know what didn't match.

### Q11 · EASY · SQL twin: d3b Q1

Merge `accounts_d` with `customers` to show each account's `acct_id`, `acct_type`, `cust_id` and the customer's `age`. Print the row count before and after. Which `how=` did you choose, and why?

In [ ]:
# Your code here


### Q12 · AVERAGE

Merge `transactions_d` with the **raw** `accounts` (duplicates included) on `orig_acct` = `acct_id`, using `validate="many_to_one"`. What happens? Catch the error with `try/except` and print it. Then do the same merge with `accounts_d`.

💡 *Hint:* `except pd.errors.MergeError as err: print(err)`.

In [ ]:
# Your code here


### Q13 · AVERAGE · SQL twin: d3b Q6

Every customer and any accounts they hold, including customers with none. Use `indicator=True`. How many customers have no accounts?

In [ ]:
# Your code here


### Q14 · AVERAGE · SQL twin: d3b Q9

Accounts in `accounts_d` that have **never received** a transaction (in `transactions_d`). How many?

💡 *Hint:* Left-merge with the distinct `dest_acct` values and keep `_merge == "left_only"`, or use `~accounts_d["acct_id"].isin(...)`.

In [ ]:
# Your code here


### Q15 · AVERAGE · SQL twin: d3b Q10

Transactions in `transactions_d` that generated **no** alerts. How many?

In [ ]:
# Your code here


### Q16 · AVERAGE · SQL twin: Day 4 inflation demo

Left-merge `alerts` onto `transactions_d` on `tx_id`. How many rows do you get, and what happens to the amount total?

In [ ]:
# Your code here


✍️ **Explain the extra rows and the extra money in one sentence.**

*Your answer:*

### Q17 · DIFFICULT · SQL twin: 04 SQL, alerts_per_tx

Fix Q16 the way the SQL did: aggregate `alerts` to **one row per `tx_id`** first (`alert_count`, and `rules_fired` as text joined with `' | '`), then merge with `validate="one_to_one"`. Prove that rows and the amount total are unchanged, and that alerts still add up to 2,618.

💡 *Hint:* `rules_fired=("rule_name", lambda s: " | ".join(sorted(s.unique())))`. After a left merge, transactions with no alerts have NaN counts: `.fillna(0)`.

In [ ]:
# Your code here


### Q18 · AVERAGE

Merge `alerts` with `investigations` (one row per alert, `validate="one_to_one"`). Then group alerts into score bands: **Critical** (≥ 90), **High** (≥ 80), **Medium** (≥ 60), **Low**, and **Unscored**. For each band: alerts, investigated, SARs, investigation rate, and SARs per investigated alert.

Is a higher-scored alert more likely to be **investigated**? More likely to end in a **SAR**?

💡 *Hint:* `np.select([score >= 90, score >= 80, score >= 60, score.notna()], ["Critical", "High", "Medium", "Low"], default="Unscored")`. Count investigated with `("case_id", "count")`.

In [ ]:
# Your code here


✍️ **What does this table suggest about how investigators choose which alerts to work?**

*Your answer:*

### Q19 · AVERAGE · SQL twin: d3b Q5

For transactions in `transactions_d` of PHP 500,000 or more, attach the **destination** account type by merging `dest_acct` to `accounts_d`. How many stay inside the bank, and how many leave it?

In [ ]:
# Your code here


### Q20 · AVERAGE · SQL twin: d3b Q11–Q12

Received side: for transactions that land on an internal account, show per destination `acct_type` the number of transactions, the total received amount, and the number of distinct receiving accounts.

In [ ]:
# Your code here


### Q21 · AVERAGE

Build the **customer table**: one row per `orig_cust_id` from `df`, with `tx_count`, `total_sent`, `alerts` (sum of `alert_count`) and `sars` (sum of `sar_count`). Then merge `customers` for `occupation`, `declared_income` and `kyc_risk_score`, with `validate="one_to_one"` and `indicator=True`. Call it `cust`.

In [ ]:
# Your code here


### Q22 · AVERAGE

Reconcile `cust`: does `total_sent` add up to the extract total? How many customers in `cust` have **no** record in `customers`?

In [ ]:
# Your code here


### Q23 · AVERAGE

Add `value_to_income` = `total_sent` ÷ `declared_income`, NaN when income is 0 or missing. How many customers sent **more than** their declared annual income? More than **5×**? Show the top 10.

In [ ]:
# Your code here


### Q24 · DIFFICULT

Compare customers who sent more than their declared income with everyone else: number of customers, total SARs, and each group's share of all SARs.

💡 *Hint:* `np.where(cust["value_to_income"] > 1, "sent > declared income", "rest (incl. unknown)")`, then `groupby`.

In [ ]:
# Your code here


✍️ **What does this suggest, and what can't it prove?**

*Your answer:*

### Q25 · DIFFICULT · SQL twin: d3b Q19, Q39

Classify each transaction's customer by KYC risk: **High** (≥ 80), **Elevated** (≥ 60), **Standard**, and **Unknown** when the score is missing. Then:
1. per risk category: distinct customers, transactions and alert rate;
2. a pivot table of alert rate (%) with risk categories as rows and `channel_clean` as columns.

💡 *Hint:* Put the missing-score condition first in `np.select`, or missing scores fall into the default. Pivot: `df.pivot_table(index=..., columns=..., values="was_alerted", aggfunc="mean")`.

In [ ]:
# Your code here


---
### Done?

Unfinished questions are homework. Bring the one you were most stuck on to tomorrow's opener, along with what you tried.